In [ ]:
import pandas as pd

# We are setting our Data Path 
data = r'this\is\a\path' 
df = pd.read_csv(data)

print("Dataset Size:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
display(df.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Target variable (Label) 'y', the rest of features 'X' 
X = df.drop(columns=['Label'])
y = df['Label']

# Let's Split The Data As %80 Train, %20 Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train Data Size (X_train):", X_train.shape)
print("Test Data Size (X_test):", X_test.shape)

# All variables to same scale (mean 0, variance 1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nScaling is succesful.")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Dimension Reduction via PCA
# Hedef: Verideki orijinal bilginin %95'ini koruyacak kadar bileşen (sütun) tut
pca = PCA(n_components=0.95, random_state=42)


X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"PrePca number of Column: {X_train_scaled.shape[1]}")
print(f"PostPCA number of Features: {X_train_pca.shape[1]}")

# 2. Modal Training (Random Forest)
print("\nTraining...(This may take few seconds!)")

# We are creating a forest with 100 Decision Tree
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_pca, y_train)

# 3. Test and Critique
# Predicting with Test Data which is not trained before
y_pred = rf_model.predict(X_test_pca)

# Let's see how succesful it works
accuracy = accuracy_score(y_test, y_pred)
print(f"\nGeneral Accuracy: %{accuracy * 100:.2f}")

print("\nDetailed Classification ")
print(classification_report(y_test, y_pred))

In [ ]:
import joblib
import os

# Creating modals folder to keep all clean
os.makedirs('../models', exist_ok=True)

# 1. Save the Scaler
joblib.dump(scaler, '../models/eeg_scaler.pkl')

# 2. Save the PCA Converter 
joblib.dump(pca, '../models/eeg_pca.pkl')

# 3. Save the Trained Random Forest Modal 
joblib.dump(rf_model, '../models/eeg_rf_model.pkl')

print("Great! The Whole Pipeline is succesfully saved to 'models' folder .")